# A Hierarchical State-Switching Model of Honeybee Waggle-Dances
---

## Details

&ensp;AUTHOR:&emsp;   James Foster &emsp; 2026 10 20

MODIFIED:&emsp; James Foster &emsp; 2026 10 21

DESCRIPTION:&emsp; Each dance is assigned to one of three latent states: **forward** (waggle runs towards the dance's direction), **reversed** (waggle runs in the opposite direction) or **axial** (waggle runs in both directions).
The probability of each state is modelled as a function of condition, with individual-level random effects, and the state of each dance is marginalised out of the likelihood.
All direction parameters are axial (restricted to $\pm 90°$), so that a 180° change in direction can only be described by a change of state.
This removes the label switching between "reversed" and "rotated by 180°" that affected the mixture model in [HS_circGLMM_mixture_lowel.ipynb](HS_circGLMM_mixture_lowel.ipynb).
The analysis can be run for low (sunset), medium or high solar elevations (`elevation` below).
Modified from [HS_circGLMM_mixture_lowel.ipynb](HS_circGLMM_mixture_lowel.ipynb), the locust mixture model in Jake Graving's `unwrap` package, and the von Mises–uniform mixture for the Goldshtein bat data, written directly in `PyMC`.

INPUTS:&emsp;    `0Data/hs_dance_reorg.csv`; optionally `2Results/unwrap_models/HS_lowel_VH_*.nc` (for comparison with the `Bambi` models at low elevation)

OUTPUTS:&emsp;  `0Data/hs_dance_states_<elevation>.feather`, `2Results/state_models/HS_states_<elevation>_<fit_tag>_*.nc`, `Plots/HS_states_<elevation>_*.pdf`

REFERENCES:
- Sayin S, ... Graving JM, $\textit{et~al}$. (2025)
The behavioral mechanisms governing collective motion in swarming locusts.
$\textit{Science}$ 387,995-1000
DOI:[10.1126/science.adq7832](https://doi.org/10.1126/science.adq7832)

- Graving JM, Sayin S, Costelloe BR, Goldshtein A & Foster JJ.
Unwrapping circular statistics with Bayesian multilevel models.

- Abril-Pla, O., Andreani, V., Carroll, C., Dong, L., Fonnesbeck, C. J., Kochurov, M., Kumar, R., Lao, J., Luhmann, C. C., Martin, O. A., et al. (2023).
`PyMC`: a modern, and comprehensive probabilistic programming framework in Python.
*PeerJ Computer Science* 9, e1516.

- Seyboldt, A., Carlson, E. L., & Carpenter, B. (2026).
Preconditioning Hamiltonian Monte Carlo by minimizing Fisher Divergence.
*arXiv* preprint arXiv:2603.18845.

- Vehtari, A., Gelman, A., & Gabry, J. (2017).
Practical Bayesian model evaluation using leave-one-out cross-validation and WAIC.
*Statistics and Computing* 27, 1413-1432.

---

# First set up the required packages
Same environment as [HS_circGLMM_fit_4cond.ipynb](HS_circGLMM_fit_4cond.ipynb) (`HS-env`); install it from `nutpie_unwrap_requirements.txt` if needed:

`"{sys.executable}" -m pip install -r nutpie_unwrap_requirements.txt`

In [ ]:
%set_env PYTENSOR_FLAGS=blas__ldflags='-llapack -lblas -lcblas'

## Settings
- `elevation`: which solar elevations to analyse, as in `HS_ExploratoryPlots.Rmd`: `'low'` ($< 15°$), `'medium'` ($30–45°$) or `'high'` ($> 75°$)
- conditions included at each elevation, reference (vertical) first. At low elevation the antisolar and zenith conditions have too few individuals (1 and 4) to estimate their effects, and tilted dances are excluded throughout.
- `direction_prior`: `'bounded_vm'` (recommended: bounded dance and condition deviations, von Mises individual offsets), `'bounded'` (all deviations bounded, Normal individual offsets), `'soft'` (unbounded Normal priors) or `'vonmises'` (unbounded von Mises offsets); see *Direction* below
- `condition_bound`: `'feeder'` (recommended: each condition's direction within 60° of the feeder) or `'population'` (each condition within 60° of the population direction, as in fits before 2026 10 21, which allowed conditions up to 120° from the feeder); bounded direction priors only
- `outlier_state`: an optional state for dances whose direction is unrelated to the feeder, e.g. because the solar ephemeris failed: `'unrelated'` (recommended: runs concentrated as usual, but around a direction unrelated to the feeder), `'disoriented'` (runs uniformly distributed) or `None`
- `data_inits`: start chains near values estimated from the data, rather than at zero on the latent scale (nutpie's default)
- `outlier_effects`: whether the outlier state's probability depends on condition (successive-difference effects, as for the other states)
- `models_to_fit`: which of the models below to fit (`None` for all)
- `fit_tag`: label for the saved fits, combining the settings above
- sampler settings, shared by all models

In [ ]:
elevation = 'low' # 'low', 'medium' or 'high'
# how direction parameters are kept from imitating a reversal (see 'Direction' below):
# 'bounded_vm' (recommended), 'bounded', 'soft' or 'vonmises'
direction_prior = 'bounded_vm'
condition_bound = 'feeder' # 'feeder' or 'population' (bounded direction priors only)
outlier_state = 'unrelated' # 'unrelated', 'disoriented' or None
outlier_effects = True # outlier-state probability depends on condition
data_inits = True # start chains near data-based values
models_to_fit = ['three_states_condition'] # or None for all models
fit_tag = (direction_prior + ('_feederbound' if condition_bound == 'feeder' else '')
           + (f'_{outlier_state}' + ('_cond' if outlier_effects else '') if outlier_state else ''))

elevation_ranges = {'low': (-90, 15), 'medium': (30, 45), 'high': (75, 90)} # degrees, exclusive
conditions_by_elevation = {
    'low': ['vertical', 'horizontal'],
    'medium': ['vertical', 'horizontal', 'antisolar', 'solar'],
    'high': ['vertical', 'horizontal', 'antisolar', 'solar'],
}

sampler = "nutpie" # or "pymc"
n_draws = 1000
n_tune = 1000
n_chains = 8 # more chains, so that chains stuck in poor local optima can be screened out
target_accept = 0.95
random_seed = 20261020
logp_threshold = 20 # flag chains whose mean log posterior is this far below the best chain
screen = not data_inits # drop flagged chains (otherwise only report them)

# Prepare the data
## Import and format the data
Waggle-run angles relative to the feeder bearing (`feeder_angle`) are already in radians.
Condition effects are coded as successive differences, as in `HS_circGLMM_fit_4cond.ipynb`, progressing vertical → horizontal → antisolar → solar:

| condition | H | A | S |
|---|---|---|---|
| vertical | 0 | 0 | 0 |
| horizontal | 1 | 0 | 0 |
| antisolar | 1 | 1 | 0 |
| solar | 1 | 1 | 1 |

- `H`: effect of horizontal orientation (horizontal − vertical)
- `A`: effect of antisolar skylight (antisolar − horizontal)
- `S`: effect of solar skylight (solar − antisolar)

Only the effects needed for the included conditions are used.

In [ ]:
import os
import numpy as np
import pandas as pd

# Load the file
df_all = pd.read_csv('0Data/hs_dance_reorg.csv', sep=',')

conditions_incl = conditions_by_elevation[elevation]
el_min, el_max = elevation_ranges[elevation]
df = df_all[
    (df_all['sun_el'] > el_min) &
    (df_all['sun_el'] < el_max) &
    (df_all['condition'].isin(conditions_incl))
].copy()

# Successive-difference coding: vertical -> horizontal -> antisolar -> solar
cond_effects = {'vertical': [], 'horizontal': ['H'], 'antisolar': ['H', 'A'], 'solar': ['H', 'A', 'S']}
effect_names = [e for e in ['H', 'A', 'S'] if any(e in cond_effects[c] for c in conditions_incl)]
# Condition design matrix (conditions x effects)
X_cond = np.array([[float(e in cond_effects[c]) for e in effect_names] for c in conditions_incl])

# Angles relative to the feeder, between -pi and pi; drop missing angles
df['angle'] = np.angle(np.exp(1j * pd.to_numeric(df['feeder_angle'], errors='coerce')))
df = df.dropna(subset=['angle'])
df = df.rename(columns={'ID': 'Individual_label', 'ID_d': 'Dance_label'})
df['condition'] = pd.Categorical(df['condition'], categories=conditions_incl)
df = df.sort_values(['Dance_label', 'waggle_run']).reset_index(drop=True)

# Integer codes (0-based) for dances, individuals and conditions
df['Dance'] = df['Dance_label'].astype('category').cat.codes
df['Individual'] = df['Individual_label'].astype('category').cat.codes
df['cond_idx'] = df['condition'].cat.codes

# One row per dance
dances = (df.groupby('Dance')
            .agg(Dance_label=('Dance_label', 'first'),
                 Individual=('Individual', 'first'),
                 Individual_label=('Individual_label', 'first'),
                 condition=('condition', 'first'),
                 cond_idx=('cond_idx', 'first'),
                 n_runs=('angle', 'size'),
                 sun_el=('sun_el', 'mean'))
            .reset_index())
n_dance = len(dances)
n_ind = df['Individual'].nunique()

#save to align with raw data in R
df[['angle', 'Individual', 'Individual_label', 'Dance', 'Dance_label', 'condition', 'sun_el']].to_feather(
    f'0Data/hs_dance_states_{elevation}.feather')

print(f"Elevation: {elevation} ({el_min}° < sun_el < {el_max}°), effects: {effect_names}")
print(df.groupby('condition', observed=True).agg(runs=('angle', 'size'),
                                                 dances=('Dance', 'nunique'),
                                                 individuals=('Individual', 'nunique')))
print('Runs per dance:', dances['n_runs'].describe()[['min', '25%', '50%', '75%', 'max']].to_dict())

## Inspect the data
For each dance, the mean resultant length of the waggle-run angles ($R_1$) and of the doubled angles ($R_2$).
Unimodal dances have high $R_1$ (and $R_2$), axial dances have low $R_1$ but high $R_2$, and diffuse dances have both low.
Unimodal dances are coloured by whether their mean direction is towards the feeder (within 90°) or away from it.
Dances with fewer than 4 runs are shown as open symbols.

In [ ]:
import matplotlib.pyplot as plt

# Colours for each condition, as in HS_ExploratoryPlots.Rmd
cond_cols = {'vertical': 'darkred', 'horizontal': 'darkgreen', 'antisolar': '#008B8B', 'solar': 'darkblue'}
state_cols = {'forward': '#1b7837', 'reversed': '#762a83', 'axial': '#e08214', 'disoriented': '#808080', 'unrelated': '#808080'}

def dance_moments(a):
    z1, z2 = np.exp(1j * a).mean(), np.exp(2j * a).mean()
    return pd.Series({'R1': np.abs(z1), 'mu1': np.angle(z1), 'R2': np.abs(z2)})

dances = dances.join(df.groupby('Dance')['angle'].apply(dance_moments).unstack(), on='Dance')

fig, axes = plt.subplots(1, len(conditions_incl), figsize=(4 * len(conditions_incl), 4), sharex=True, sharey=True)
for ax, cond in zip(np.atleast_1d(axes), conditions_incl):
    dc = dances[dances['condition'] == cond]
    towards = np.cos(dc['mu1']) > 0
    for sel, col, lab in [(towards, state_cols['forward'], 'mean towards feeder'),
                          (~towards, state_cols['reversed'], 'mean away from feeder')]:
        many = sel & (dc['n_runs'] >= 4)
        few = sel & (dc['n_runs'] < 4)
        ax.scatter(dc.loc[many, 'R1'], dc.loc[many, 'R2'], color=col, alpha=0.7, label=lab)
        ax.scatter(dc.loc[few, 'R1'], dc.loc[few, 'R2'], facecolor='none', edgecolor=col, alpha=0.7)
    ax.plot([0, 1], [0, 1], color='gray', lw=0.5)
    ax.set_title(f"{cond} ({len(dc)} dances)")
    ax.set_xlabel('$R_1$ (unimodal)')
np.atleast_1d(axes)[0].set_ylabel('$R_2$ (axial)')
np.atleast_1d(axes)[0].legend(loc='lower right', fontsize='small')
plt.tight_layout()
plt.show()

# The state-switching model
## Likelihood
Each dance $d$ has a direction $\mu_d$ and concentration $\kappa_d$, and is in one of three states, $s_d$.
Given its state, the waggle runs $\theta_{dj}$ of that dance are independent:

| state | density of each waggle run |
|---|---|
| forward | $\mathcal{M}(\theta \mid \mu_d, \kappa_d)$ |
| reversed | $\mathcal{M}(\theta \mid \mu_d + \pi, \kappa_d)$ |
| axial | $w\,\mathcal{M}(\theta \mid \mu_d, \kappa_d) + (1-w)\,\mathcal{M}(\theta \mid \mu_d + \pi, \kappa_d)$ |
| unrelated (optional) | $\mathcal{M}(\theta \mid \nu_d, \kappa_d)$, $\nu_d \sim \text{Uniform}(-\pi, \pi)$ |
| disoriented (alternative) | $1 / 2\pi$ |

The state is unknown, so it is marginalised out for each dance:

$$
p(\boldsymbol{\theta}_d) = \sum_{s} \pi_{ds} \prod_j f_s(\theta_{dj} \mid \mu_d, \kappa_d)
$$

The posterior probability that dance $d$ is in each state, $p(s_d \mid \boldsymbol{\theta}_d)$, is saved for each draw.

In the axial state, the proportion of runs towards $\mu_d$ is restricted to $0.25 < w < 0.75$ (each mode accounts for at least 25% of runs, as for the `circMLE` bimodal fits in `HS_ExploratoryPlots.Rmd`), so that the axial state cannot imitate the forward or reversed states.

The optional outlier state describes dances whose direction is unrelated to the feeder, for example where the solar ephemeris failed (near the zenith, a small error in time or position gives a large error in solar azimuth). Without it, a single such dance beyond the offset bound (below) can pull its condition's direction towards it: at high elevation, one vertical dance at −72° (sun elevation 84°) moved the vertical direction from about +14° to −7°, or in some chains was described as reversed.
A uniform (disoriented) state described this dance poorly, because its runs are tightly clustered ($R = 0.87$), just not around the feeder, so chains still alternated between that and a forward dance at −57°.
The unrelated state instead keeps the dance's concentration but gives it a direction $\nu_d$ unrelated to the feeder. Marginalising over $\nu_d$ gives a closed form without extra parameters, $p(\boldsymbol{\theta}_d) = I_0(\kappa_d R_d) \,/\, \left(2\pi I_0(\kappa_d)\right)^{n_d}$, where $R_d = \left|\sum_j e^{i\theta_{dj}}\right|$.

## Direction ($\mu$)
$\mu_d = \mu_{c[d]} + B_\text{dance} \tanh\left(\frac{u_{i[d]} + v_d}{B_\text{dance}}\right)$, with $\mu_c = \mu_0 + B_\text{cond} \tanh\left(\frac{\sum_e X_{ce} \beta^\mu_e}{B_\text{cond}}\right)$

where $u_i \sim \mathcal{N}(0, \sigma_u)$ and $v_d \sim \mathcal{N}(0, \sigma_v)$ are individual and dance offsets, $\beta^\mu$ are successive-difference condition effects (on a latent scale), and $\mu_0 = B_\text{pop} \tanh(0.5 z / B_\text{pop})$, $z \sim \mathcal{N}(0, 1)$, is kept within $B_\text{pop} = 60°$ of the feeder (the vertical, reference, condition defines which direction is "forward").
The function $B \tanh(x / B)$ is approximately $x$ for small $x$ but never reaches $\pm B$, so:
- each condition's direction is within $B_\text{cond} = 60°$ of the population direction (`condition_bound = 'population'`), or, with `condition_bound = 'feeder'` (recommended), within 60° of the feeder: $\mu_c = B_\text{cond} \tanh\left(\frac{0.5 z + \sum_e X_{ce} \beta^\mu_e}{B_\text{cond}}\right)$. Bounding relative to the population direction allows a condition to be up to 120° from the feeder, and in test fits some chains settled with condition directions near −90° to −105°, i.e. a 'forward' direction pointing away from the feeder;
- each dance's direction is within $B_\text{dance} = 60°$ of its condition's direction, so that any two dances in the same condition differ by at most 120° in $\mu_d$.

A dance that points about 180° from the others must therefore be in the reversed state, rather than a forward dance with a large offset, and similarly for conditions.
(Bounding the individual and dance offsets separately, or at 90°, is not enough: offsets saturating at $+90°$ and $-90°$ recreate a 180° difference, and in test fits some chains settled on this configuration.
Likewise, with population and condition directions allowed up to 90°, single chains settled on a population direction near $-75°$, or a "forward" solar direction near the $-90°$ bound, partly imitating a reversal.)

**Alternative without bounds** (`direction_prior = 'soft'`): $\mu_d = \mu_0 + \sum_e X_{ce}\beta^\mu_e + u_{i[d]} + v_d$, with $\mu_0 \sim \mathcal{N}(0, 0.5)$, $\beta^\mu \sim \mathcal{N}(0, 0.5)$, and $u_i \sim \mathcal{N}(0, \sigma_u)$, $v_d \sim \mathcal{N}(0, \sigma_v)$ with $\sigma_u, \sigma_v \sim \text{Gamma}(4, 10)$ (mean 0.4 rad, $P(\sigma > 1) \approx 1\%$).
Nothing is bounded, but a 180° shift in a condition (6 SD) or in a dance (≈ 8 SD at $\sigma = 0.4$) is very unlikely a priori, so reversals should be described by states.
Unlike the mixture model, which had no reversed state and so needed large offsets to describe reversed dances, the offsets here do not need to grow to fit the data.

**Von Mises alternative** (`direction_prior = 'vonmises'`): $\mu_0 \sim \mathcal{M}(0, 5)$ (≈ 26° SD), condition effects $\beta^\mu \sim \mathcal{M}(0, 2)$, and von Mises individual and dance offsets, $u_i \sim \mathcal{M}(0, \kappa_u)$, $v_d \sim \mathcal{M}(0, \kappa_v)$.
Individual offsets are strongly concentrated, $\kappa_u \sim \text{LogNormal}(\log 30, 0.6)$, as recommended by Graving et al., which gave stable estimates of individual offsets in unimodal models.
Dance offsets are less concentrated, $\kappa_v \sim \text{LogNormal}(\log 5, 0.5)$, since dances genuinely vary in direction (an SD of 25–30° in the converged fits); even at $\kappa_v = 5$, a 180° offset is $e^{-10}$ times less likely than none.
Nothing is bounded, but strong concentration makes reversals by offsets very unlikely.
The equivalent circular SDs, $\sqrt{-2 \log A_1(\kappa)}$, are saved as `mu_Individual_sigma` and `mu_Dance_sigma` for comparison with the other options.

**Recommended combination** (`direction_prior = 'bounded_vm'`): bounded condition directions and dance offsets, as in `'bounded'`, but von Mises individual offsets with $\kappa_u \sim \text{LogNormal}(\log 30, 0.6)$, added to the dance offset before bounding.

Results of test fits that led to this choice:
- `'bounded'`: converged at low elevation after chain screening; at high elevation, one kept chain per model placed the vertical direction at +35–39° (instead of about −7°), compensated by the horizontal effect and a larger individual offset SD, i.e. the population direction traded off against weakly constrained Normal individual offsets.
- `'soft'`: two of four chains stuck, and the other two disagreed (log posterior about 5 apart), with many dances offset by > 45°.
- `'vonmises'`: individual offsets were stable ($\kappa_u \approx 30$, SD ≈ 0.2 rad in every chain), but dance concentrations collapsed ($\kappa_v \approx 1.4$–1.9, SD 50–60°) even with $\kappa_v \sim \text{LogNormal}(\log 30, 0.6)$, so dance offsets wrapped round to imitate reversals and chains disagreed on the condition directions.

So strongly concentrated von Mises offsets suit individuals, but dances (particularly at high elevation, where many are rotated 45–135° from the feeder) vary too much in direction for an unbounded offset to be kept from imitating a reversal.
A dance that points 180° from its individual's mean must therefore be in the reversed state, rather than a forward dance with a 180° offset, and similarly for individuals and conditions.

## Concentration ($\kappa$)
$\text{softplus}^{-1}(\kappa_d) = \alpha^\kappa + \sum_e X_{de} \beta^\kappa_e + \sigma^\kappa_i z_{i[d]} + \sigma^\kappa_d z_d$

## State probabilities ($\pi$)
Multinomial logit, with forward as the reference state:

The outlier state, if included, has its own probability, $\pi_{d,\text{out}} = \text{logit}^{-1}(\alpha^\text{out} + \sum_e X_{de} \beta^\text{out}_e)$ (condition effects only with `outlier_effects = True`), and the other states share the remaining $1 - \pi_{d,\text{out}}$ as above. Condition effects allow, for example, the bees' internal model of the solar ephemeris to be more error-prone under some skylight conditions than others. (An intercept-only logit against forward would make the outlier probability proportional to the forward probability, so lower in conditions with many reversed dances.)

$\log(\pi_{d,\text{reversed}} / \pi_{d,\text{forward}}) = \alpha^\text{rev} + \sum_e X_{de} \beta^\text{rev}_e + \sigma^\text{rev} z^\text{rev}_{i[d]}$

$\log(\pi_{d,\text{axial}} / \pi_{d,\text{forward}}) = \alpha^\text{ax} + \sum_e X_{de} \beta^\text{ax}_e + \sigma^\text{ax} z^\text{ax}_{i[d]}$

The effects of condition on these logits ($\beta^\text{rev}$, $\beta^\text{ax}$) test whether a condition makes reversed or axial dances more likely.

In [ ]:
import pymc as pm
import pytensor.tensor as pt
import arviz as az
import xarray as xr

from scipy.special import logit

OUTLIER_STATES = ('unrelated', 'disoriented') # states whose probability does not depend on condition
STATES = ['forward', 'reversed', 'axial'] + ([outlier_state] if outlier_state else [])
LOG_2PI = np.log(2 * np.pi)


def inverse_softplus(x):
    return np.log(np.expm1(x))


def vonmises_lognorm(kappa):
    """Log normalising constant of the von Mises density, log(2*pi*I0(kappa)), stable for large kappa."""
    return LOG_2PI + kappa + pt.log(pt.ive(0, kappa))


MAX_POP = np.pi / 3  # maximum deviation of the population (reference) direction from the feeder
MAX_COND = np.pi / 3  # maximum deviation of a condition's direction from the feeder or the population direction
MAX_OFFSET = np.pi / 3  # maximum deviation of a dance's direction from its condition's direction
MAX_SIGMA = 0.6  # maximum SD (radians) of individual and dance offsets, so sqrt(2)*MAX_SIGMA < MAX_OFFSET


def circ_sd(kappa):
    """Circular standard deviation of a von Mises distribution, sqrt(-2 log A1(kappa))."""
    return pt.sqrt(-2 * pt.log(pt.ive(1, kappa) / pt.ive(0, kappa)))


def bounded(x, bound):
    """bound*tanh(x/bound): approximately x for small x, but never reaching +/-bound."""
    return bound * pt.tanh(x / bound)


def build_state_model(df, dances, states=STATES, state_effects=True, direction=None, fixed_directions=False):
    """
    Hierarchical state-switching model of waggle-run angles.

    states:        subset of STATES to include (forward is always the reference state)
    state_effects: whether the state probabilities depend on condition
    direction:     'bounded_vm', 'bounded', 'soft' or 'vonmises' direction priors (default: `direction_prior`)
    fixed_directions: fix every condition's forward direction at the feeder (bounded priors only), so that
                  direction varies only between individuals and dances
    """
    direction = direction or direction_prior
    n_eff = len(effect_names)
    X_d = X_cond[dances['cond_idx'].values] # dances x effects
    ind_idx = dances['Individual'].values
    dance_idx = df['Dance'].values
    # Sufficient statistics for each dance: number of runs and sums of cos and sin of the run angles
    # (the forward and reversed states depend on the runs only through these)
    n_d = np.bincount(dance_idx, minlength=n_dance).astype(float)
    C_d = np.bincount(dance_idx, weights=np.cos(df['angle'].values), minlength=n_dance)
    S_d = np.bincount(dance_idx, weights=np.sin(df['angle'].values), minlength=n_dance)
    # dance x run indicator matrix, to sum the axial-state log-densities within each dance
    M = np.zeros((n_dance, len(df)))
    M[dance_idx, np.arange(len(df))] = 1.0
    coords = {'dance': dances['Dance_label'].values,
              'individual': np.sort(dances['Individual_label'].unique()),
              'condition': conditions_incl,
              'effect': effect_names,
              'state': list(states),
              'other_state': [s for s in states if s != 'forward']}

    with pm.Model(coords=coords) as model:
        theta = pm.Data('theta', df['angle'].values)

        # --- Direction ---
        if direction in ('bounded', 'bounded_vm'):
            # bounded deviations, so that 180° changes can only be changes of state
            if not fixed_directions:
                mu0_z = 0.5 * pm.Normal('mu_Intercept_z', 0.0, 1.0)
                mu0 = pm.Deterministic('mu_Intercept', bounded(mu0_z, MAX_POP))
            if fixed_directions:
                # forward direction of every condition at the feeder (reversed at 180°, axial at 0° and 180°)
                mu_c = pm.Deterministic('mu_condition', pt.zeros(len(conditions_incl)), dims='condition')
            elif n_eff and condition_bound == 'feeder':
                # successive-difference effects on a latent scale; each condition within 60° of the feeder
                mu_eff = pm.Normal('mu_effect', mu=0.0, sigma=1.0, dims='effect')
                mu_c = pm.Deterministic('mu_condition', bounded(mu0_z + pt.dot(X_cond, mu_eff), MAX_COND),
                                        dims='condition')
            elif n_eff:
                # successive-difference effects on a latent scale; each condition within 60° of the population
                mu_eff = pm.Normal('mu_effect', mu=0.0, sigma=1.0, dims='effect')
                mu_c = pm.Deterministic('mu_condition', mu0 + bounded(pt.dot(X_cond, mu_eff), MAX_COND),
                                        dims='condition')
            else:
                mu_c = pm.Deterministic('mu_condition', pt.stack([mu0]), dims='condition')
            # individual and dance offsets (non-centred), whose sum is kept within 60° of the condition.
            # Their SDs are capped (MAX_SIGMA), so that the combined SD stays below the bound: otherwise the
            # bounded offsets pile up at +/-60°, which recreates a choice between two distant directions
            if direction == 'bounded_vm':
                # individuals: von Mises, strongly concentrated on small offsets (as in Graving et al.)
                kappa_mu_ind = pm.LogNormal('mu_Individual_kappa', mu=np.log(30.0), sigma=0.6)
                pm.Deterministic('mu_Individual_sigma', circ_sd(kappa_mu_ind))
                u_ind = pm.VonMises('mu_Individual', mu=0.0, kappa=kappa_mu_ind, dims='individual')
            else:
                sigma_mu_ind = pm.TruncatedNormal('mu_Individual_sigma', mu=0.0, sigma=0.5, lower=0.0, upper=MAX_SIGMA)
                u_ind = sigma_mu_ind * pm.Normal('mu_Individual_z', 0.0, 1.0, dims='individual')
            sigma_mu_dance = pm.TruncatedNormal('mu_Dance_sigma', mu=0.0, sigma=0.5, lower=0.0, upper=MAX_SIGMA)
            v_dance = sigma_mu_dance * pm.Normal('mu_Dance_z', 0.0, 1.0, dims='dance')
            offset = pm.Deterministic('mu_offset', bounded(u_ind[ind_idx] + v_dance, MAX_OFFSET), dims='dance')
        elif direction == 'vonmises':
            # von Mises terms, strongly concentrated on small offsets (no bounds)
            mu0 = pm.VonMises('mu_Intercept', mu=0.0, kappa=5.0)
            if n_eff:
                mu_eff = pm.VonMises('mu_effect', mu=0.0, kappa=2.0, dims='effect')
                mu_c = pm.Deterministic('mu_condition', mu0 + pt.dot(X_cond, mu_eff), dims='condition')
            else:
                mu_c = pm.Deterministic('mu_condition', pt.stack([mu0]), dims='condition')
            # individuals: strongly concentrated (as in Graving et al.); dances: genuinely more variable
            kappa_mu_ind = pm.LogNormal('mu_Individual_kappa', mu=np.log(30.0), sigma=0.6)
            kappa_mu_dance = pm.LogNormal('mu_Dance_kappa', mu=np.log(5.0), sigma=0.5)
            pm.Deterministic('mu_Individual_sigma', circ_sd(kappa_mu_ind))
            pm.Deterministic('mu_Dance_sigma', circ_sd(kappa_mu_dance))
            u_ind = pm.VonMises('mu_Individual', mu=0.0, kappa=kappa_mu_ind, dims='individual')
            v_dance = pm.VonMises('mu_Dance', mu=0.0, kappa=kappa_mu_dance, dims='dance')
            offset = pm.Deterministic('mu_offset', u_ind[ind_idx] + v_dance, dims='dance')
        else:
            # soft: unbounded Normal terms whose priors make a 180° change very unlikely
            mu0 = pm.Deterministic('mu_Intercept', 0.5 * pm.Normal('mu_Intercept_z', 0.0, 1.0))
            if n_eff:
                mu_eff = pm.Normal('mu_effect', mu=0.0, sigma=0.5, dims='effect')
                mu_c = pm.Deterministic('mu_condition', mu0 + pt.dot(X_cond, mu_eff), dims='condition')
            else:
                mu_c = pm.Deterministic('mu_condition', pt.stack([mu0]), dims='condition')
            # offset SDs: mean 0.4 rad, light upper tail (P(sigma > 1) ≈ 1%), no mass at zero
            sigma_mu_ind = pm.Gamma('mu_Individual_sigma', alpha=4.0, beta=10.0)
            sigma_mu_dance = pm.Gamma('mu_Dance_sigma', alpha=4.0, beta=10.0)
            u_ind = sigma_mu_ind * pm.Normal('mu_Individual_z', 0.0, 1.0, dims='individual')
            v_dance = sigma_mu_dance * pm.Normal('mu_Dance_z', 0.0, 1.0, dims='dance')
            offset = pm.Deterministic('mu_offset', u_ind[ind_idx] + v_dance, dims='dance')
        mu_d = pm.Deterministic('mu', mu_c[dances['cond_idx'].values] + offset, dims='dance')

        # --- Concentration ---
        k0 = pm.Normal('kappa_Intercept', mu=inverse_softplus(3.0), sigma=3.0)
        k_eff = pt.dot(X_d, pm.Normal('kappa_effect', mu=0.0, sigma=1.0, dims='effect')) if n_eff else 0.0
        sk_ind = pm.HalfNormal('kappa_Individual_sigma', sigma=1.0)
        sk_dance = pm.HalfNormal('kappa_Dance_sigma', sigma=1.0)
        zk_ind = pm.Normal('kappa_Individual_z', 0.0, 1.0, dims='individual')
        zk_dance = pm.Normal('kappa_Dance_z', 0.0, 1.0, dims='dance')
        kappa_d = pm.Deterministic('kappa',
                                   pt.softplus(k0 + k_eff + sk_ind * zk_ind[ind_idx] + sk_dance * zk_dance),
                                   dims='dance')

        # --- State probabilities: multinomial logit vs forward ---
        etas = [pt.zeros(n_dance)]
        main_states = [s for s in states if s not in OUTLIER_STATES]
        out_states = [s for s in states if s in OUTLIER_STATES]
        for s in main_states[1:]:
            a_s = pm.Normal(f'{s}_Intercept', mu=-1.0, sigma=1.5)
            b_s = (pt.dot(X_d, pm.Normal(f'{s}_effect', mu=0.0, sigma=1.5, dims='effect'))
                   if (state_effects and n_eff) else 0.0)
            s_sig = pm.HalfNormal(f'{s}_Individual_sigma', sigma=1.0)
            z_s = pm.Normal(f'{s}_Individual_z', 0.0, 1.0, dims='individual')
            etas.append(a_s + b_s + s_sig * z_s[ind_idx])
        log_pi = pt.special.log_softmax(pt.stack(etas, axis=1), axis=1)
        if out_states:
            # outlier state: its own probability (expected to be small), the others share the remainder
            s = out_states[0]
            eta_out = pm.Normal(f'{s}_Intercept', mu=-2.0, sigma=1.0) * pt.ones(n_dance)
            if outlier_effects and state_effects and n_eff:
                # successive-difference condition effects, regularised as only a few dances are expected
                eta_out = eta_out + pt.dot(X_d, pm.Normal(f'{s}_effect', mu=0.0, sigma=1.0, dims='effect'))
            # log(1 - p_out) = -softplus(eta_out), log(p_out) = -softplus(-eta_out)
            log_pi = pt.concatenate([log_pi - pt.softplus(eta_out)[:, None],
                                     -pt.softplus(-eta_out)[:, None]], axis=1)
        log_pi = pm.Deterministic('log_pi', log_pi, dims=('dance', 'state'))

        # --- Log-likelihood of each dance's runs, for each state ---
        # forward and reversed: sum_j kappa*cos(theta_j - mu) = kappa*(C*cos(mu) + S*sin(mu))
        norm_d = n_d * vonmises_lognorm(kappa_d) # computed once per dance
        proj_d = kappa_d * (C_d * pt.cos(mu_d) + S_d * pt.sin(mu_d))
        L = {'forward': proj_d - norm_d, 'reversed': -proj_d - norm_d}
        if 'axial' in states:
            # axial: each run is a mixture of the two directions, so needs the individual runs
            w_ax = pm.Deterministic('w_axial', 0.25 + 0.5 * pm.math.sigmoid(pm.Normal('w_axial_logit', 0.0, 1.0)))
            proj_r = kappa_d[dance_idx] * pt.cos(theta - mu_d[dance_idx])
            L['axial'] = pt.dot(M, pt.logaddexp(pt.log(w_ax) + proj_r, pt.log1p(-w_ax) - proj_r)) - norm_d
        if 'disoriented' in states:
            L['disoriented'] = pt.as_tensor(-n_d * LOG_2PI)
        if 'unrelated' in states:
            # von Mises runs around a direction unrelated to the feeder, marginalised over a uniform direction:
            # I0(kappa*R) / (2*pi*I0(kappa))^n, with R the length of the dance's resultant vector
            R_d = np.hypot(C_d, S_d)
            L['unrelated'] = kappa_d * R_d + pt.log(pt.ive(0, kappa_d * R_d)) - norm_d
        L_ds = pt.stack([L[s] for s in states], axis=1) # dances x states

        # --- Marginalise the state of each dance ---
        joint = log_pi + L_ds
        loglik = pm.Deterministic('loglik_dance', pt.logsumexp(joint, axis=1), dims='dance')
        pm.Deterministic('p_state', pt.special.softmax(joint, axis=1), dims=('dance', 'state'))
        pm.Potential('obs', loglik.sum())
    return model


def data_initvals(model):
    """
    Starting values estimated from the data (bounded direction priors): each condition's direction from the
    axes of its dances (doubled-angle means, weighted by R2; the end within 90° of the feeder is forward),
    and the concentration from the dances' mean resultant lengths. Other parameters start at their prior
    means or at moderate values. nutpie jitters each chain's start around these values.
    """
    z2 = df.groupby('Dance')['angle'].apply(lambda a: np.exp(2j * a).mean()).values
    cond_d = dances['condition'].values
    axis_c = np.array([np.angle(z2[cond_d == c].sum()) / 2 for c in conditions_incl])
    # latent condition directions (inverse of the bounding function), kept away from the bounds
    lat_c = MAX_COND * np.arctanh(np.clip(axis_c / MAX_COND, -0.8, 0.8))
    R = np.clip(np.maximum(dances['R1'].values, np.abs(z2)), 0.05, 0.95)
    kappa_hat = np.median(R * (2 - R ** 2) / (1 - R ** 2)) # Banerjee et al. (2005) approximation
    iv = {'mu_Intercept_z': 2 * lat_c[0] if condition_bound == 'feeder' else 2 * MAX_POP * np.arctanh(np.clip(axis_c[0] / MAX_POP, -0.8, 0.8)),
          'mu_Dance_sigma': 0.3, 'mu_Individual_kappa': 30.0, 'mu_Individual_sigma': 0.2,
          'kappa_Intercept': inverse_softplus(kappa_hat),
          'kappa_Individual_sigma': 0.5, 'kappa_Dance_sigma': 0.5,
          'reversed_Individual_sigma': 0.5, 'axial_Individual_sigma': 0.5,
          'unrelated_Intercept': -2.0, 'disoriented_Intercept': -2.0}
    if effect_names:
        if condition_bound == 'feeder':
            eff = np.linalg.lstsq(X_cond, lat_c - lat_c[0], rcond=None)[0]
        else:
            rel = MAX_COND * np.arctanh(np.clip((axis_c - axis_c[0]) / MAX_COND, -0.8, 0.8))
            eff = np.linalg.lstsq(X_cond, rel, rcond=None)[0]
        iv['mu_effect'] = eff
    return {k: v for k, v in iv.items() if k in model.named_vars and model[k] in model.free_RVs}


def init_mean(model):
    """The data-based starting values on nutpie's unconstrained scale (free variables in model order)."""
    for name, val in data_initvals(model).items():
        model.set_initval(model[name], np.asarray(val, dtype=float))
    ip = model.initial_point()
    return np.concatenate([np.ravel(ip[model.rvs_to_values[rv].name]) for rv in model.free_RVs])


def screen_chains(idata, threshold=logp_threshold, drop=True):
    """
    Drop chains stuck in poor local optima: those whose mean log posterior density is more than
    `threshold` below that of the best chain. A difference of 20 corresponds to a density ratio of
    e^-20, so such chains contribute negligible posterior mass, but would distort the summaries.
    """
    lp_name = 'logp' if 'logp' in idata.sample_stats else 'lp' # nutpie or PyMC
    lp = idata.sample_stats[lp_name].mean('draw')
    keep = (lp >= lp.max() - threshold).values
    print(f"Mean log posterior per chain: {np.round(lp.values, 1)}")
    if not keep.all():
        print(f"{'Dropping' if drop else 'Flagged (not dropped):'} chains {np.where(~keep)[0].tolist()} "
              f"(more than {threshold} below the best); keeping {keep.sum() if drop else len(keep)} of {len(keep)}")
    if not drop:
        return idata
    idata = idata.isel(chain=np.where(keep)[0])
    # renumber the remaining chains
    return idata.assign_coords(chain=np.arange(keep.sum()),
                               groups=[g for g in idata.groups() if 'chain' in idata[g].dims])


idatas_all = {} # unscreened fits, to compare kept and dropped chains


def fit_state_model(model, label):
    """Sample, screen chains, attach the dance-level log-likelihood (for leave-one-dance-out LOO) and save."""
    start = {}
    if data_inits and direction_prior in ('bounded', 'bounded_vm'):
        start = ({'nuts_sampler_kwargs': {'init_mean': init_mean(model)}} if sampler == 'nutpie'
                 else {'initvals': data_initvals(model)})
    with model:
        idata = pm.sample(draws=n_draws, tune=n_tune, chains=n_chains,
                          nuts_sampler=sampler, target_accept=target_accept,
                          random_seed=random_seed, **start)
    idatas_all[label] = idata
    idata = screen_chains(idata, drop=screen)
    idata.add_groups(log_likelihood=xr.Dataset({'loglik_dance': idata.posterior['loglik_dance']}))
    os.makedirs("2Results/state_models", exist_ok=True)
    az.to_netcdf(idata, f"2Results/state_models/HS_states_{elevation}_{fit_tag}_{label}.nc")
    return idata

## Priors
- Population direction $\mu_0$: approximately $\mathcal{N}(0, 0.5)$ rad, bounded at $\pm 60°$; latent condition effects $\beta^\mu \sim \mathcal{N}(0, 1)$ (weak), bounded at $\pm 60°$.
- Individual and dance offsets: SDs $\sigma_u, \sigma_v \sim \text{HalfNormal}(0.5)$ rad, truncated at 0.6 rad (34°), wide enough for the dance-to-dance drift seen in the mixture model.
The cap keeps the combined SD ($\le 0.85$ rad) below the 60° bound; with larger SDs, the bounded offsets pile up at $\pm 60°$, which recreates a choice between two distant directions (in test fits, some chains settled on this configuration).
- State logits: intercepts $\mathcal{N}(-1, 1.5)$ (forward most likely a priori), condition effects $\mathcal{N}(0, 1.5)$, individual SD $\text{HalfNormal}(1)$. These keep the logits finite when a state is never observed in a condition (e.g. no reversed vertical dances).
- Axial weight: $\text{logit}((w - 0.25) / 0.5) \sim \mathcal{N}(0, 1)$, centred on $w = 0.5$.

## Models compared
0. **Unimodal** (forward only; null model, as in the `GUVdances` analysis), with unbounded von Mises direction priors (`direction = 'vonmises'`), so that reversed dances can be described by large individual and dance offsets, and axial dances by low dance-level concentration.
1. **Two states** (forward, reversed), state probabilities depend on condition.
2. **Three states**, state probabilities the same in all conditions.
3. **Three states**, state probabilities depend on condition.
4. **Three states, fixed directions**: as 3, but every condition's forward direction is fixed at the feeder (reversed at 180°, axial at 0° and 180°), so that direction varies only between individuals and dances. Comparing 3 with 4 tests whether population-level shifts in direction explain the data better than individual and dance offsets.

Comparing 3 with 2 tests whether axial dances are needed; comparing 3 with 1 tests whether condition affects which states dances are in.

In [ ]:
models = {
    'unimodal': build_state_model(df, dances, states=['forward'], state_effects=False, direction='vonmises'),
    'two_states': build_state_model(df, dances, states=['forward', 'reversed'] + STATES[3:], state_effects=True),
    'three_states_same': build_state_model(df, dances, states=STATES, state_effects=False),
    'three_states_condition': build_state_model(df, dances, states=STATES, state_effects=True),
    'three_states_fixed': build_state_model(df, dances, states=STATES, state_effects=True, fixed_directions=True),
}
dis_label = f'/{outlier_state}' if outlier_state else ''
model_labels = {
    'unimodal': 'Unimodal (forward only, unbounded offsets)',
    'two_states': f'Forward/reversed{dis_label}, condition-dependent',
    'three_states_same': f'Forward/reversed/axial{dis_label}, same in all conditions',
    'three_states_condition': f'Forward/reversed/axial{dis_label}, condition-dependent',
    'three_states_fixed': f'Forward/reversed/axial{dis_label}, condition-dependent, directions fixed at feeder',
}
if models_to_fit is not None:
    models = {k: v for k, v in models.items() if k in models_to_fit}
models['three_states_condition']

# Prior predictive check
Before fitting, check what the priors imply, for each direction prior, using the condition-dependent three-state model.
Since chain screening showed that some chains settle in configurations that seem biologically implausible, the question here is how much prior probability those configurations have.
If the priors give them substantial probability, and we would have rejected them before seeing the data, the priors are too permissive and can be tightened on those grounds (rather than because of where chains went in this dataset).

## Prior probability of implausible configurations

In [ ]:
def wrap(x):
    return np.angle(np.exp(1j * x))

def sample_prior(direction, n=1000):
    """Draws of the main parameters from the prior (the likelihood, a Potential, is ignored)."""
    m = build_state_model(df, dances, states=STATES, state_effects=True, direction=direction)
    vs = ['mu_condition', 'mu_offset', 'mu_Individual_sigma', 'mu_Dance_sigma', 'mu', 'kappa', 'log_pi', 'w_axial']
    pr = pm.sample_prior_predictive(draws=n, model=m, var_names=vs, random_seed=random_seed).prior
    return {v: pr[v].values.reshape(n, *pr[v].shape[2:]) for v in vs}

priors_sampled = {d: sample_prior(d) for d in ['bounded_vm', 'bounded', 'soft', 'vonmises']}

def implausibility(pr):
    mu_c = np.rad2deg(wrap(pr['mu_condition'])) # draws x conditions
    off = np.abs(np.rad2deg(wrap(pr['mu_offset']))) # draws x dances
    res = {
        '|vertical direction| > 20°': np.mean(np.abs(mu_c[:, 0]) > 20),
        '|vertical direction| > 45°': np.mean(np.abs(mu_c[:, 0]) > 45),
    }
    if len(conditions_incl) > 1:
        shift = np.abs(np.rad2deg(wrap(np.deg2rad(mu_c[:, 1:] - mu_c[:, [0]]))))
        res['any condition > 45° from vertical'] = np.mean(np.any(shift > 45, axis=1))
        res['any condition > 90° from vertical'] = np.mean(np.any(shift > 90, axis=1))
    res.update({
        'individual offset SD > 0.6 rad (34°)': np.mean(pr['mu_Individual_sigma'] > 0.6),
        'dance offset SD > 0.6 rad (34°)': np.mean(pr['mu_Dance_sigma'] > 0.6),
        'dance offset SD > 1 rad (57°)': np.mean(pr['mu_Dance_sigma'] > 1),
        'mean proportion of dances offset > 45°': np.mean(off > 45),
        'mean proportion of dances offset > 90°': np.mean(off > 90),
    })
    return pd.Series(res).round(3)

pd.DataFrame({d: implausibility(pr) for d, pr in priors_sampled.items()})

## Simulated data from the prior
Datasets simulated from the prior, with the same dances and numbers of runs as the observed data, compared with the observed data.
For each dance: the proportion of waggle runs within 90° of the feeder, and the mean resultant length ($R_1$).
Simulated distributions (thin lines, 50 datasets) should cover the observed ones (thick line) without being much wider than plausible.

In [ ]:
rng = np.random.default_rng(random_seed)
dance_of_run = df['Dance'].values

def simulate_runs(pr, k):
    """Simulate run angles for prior draw k: a state for each dance, then von Mises runs."""
    p = np.exp(pr['log_pi'][k]) # dances x states
    state = np.array([rng.choice(len(STATES), p=pi / pi.sum()) for pi in p])
    mu, kappa = pr['mu'][k][dance_of_run], pr['kappa'][k][dance_of_run]
    s_run = state[dance_of_run]
    # forward: mu; reversed: mu + pi; axial: mu + pi with probability 1 - w
    flip = (s_run == 1) | ((s_run == 2) & (rng.random(len(df)) > pr['w_axial'][k]))
    angles = rng.vonmises(mu + np.pi * flip, kappa)
    if 'disoriented' in STATES:
        dis = s_run == STATES.index('disoriented')
        angles[dis] = rng.uniform(-np.pi, np.pi, dis.sum())
    if 'unrelated' in STATES:
        # a random direction for each unrelated dance
        nu = rng.uniform(-np.pi, np.pi, n_dance)[dance_of_run]
        unr = s_run == STATES.index('unrelated')
        angles[unr] = rng.vonmises(nu[unr], kappa[unr])
    return wrap(angles)

def dance_summaries(angles):
    d = pd.DataFrame({'Dance': dance_of_run, 'angle': angles})
    g = d.groupby('Dance')['angle']
    return pd.DataFrame({'prop_feeder': g.apply(lambda a: np.mean(np.cos(a) > 0)),
                         'R1': g.apply(lambda a: np.abs(np.exp(1j * a).mean()))})

obs_summ = dance_summaries(df['angle'].values)
bins = np.linspace(0, 1, 11)
fig, axes = plt.subplots(len(priors_sampled), 2 * len(conditions_incl),
                         figsize=(5 * len(conditions_incl), 3 * len(priors_sampled)), sharex=True)
for r, (dprior, pr) in enumerate(priors_sampled.items()):
    sims = [dance_summaries(simulate_runs(pr, k)) for k in rng.choice(len(pr['mu']), 50, replace=False)]
    for c, cond in enumerate(conditions_incl):
        sel = (dances['condition'] == cond).values
        for j, stat in enumerate(['prop_feeder', 'R1']):
            ax = axes[r, 2 * c + j]
            for sm in sims:
                h, _ = np.histogram(sm[stat].values[sel], bins=bins)
                ax.stairs(h / sel.sum(), bins, color=cond_cols[cond], alpha=0.1)
            h, _ = np.histogram(obs_summ[stat].values[sel], bins=bins)
            ax.stairs(h / sel.sum(), bins, color='black', lw=2)
            ax.set_title(f"{dprior} prior: {cond}", fontsize='small')
            if r == len(priors_sampled) - 1:
                ax.set_xlabel({'prop_feeder': 'proportion of runs within 90° of feeder', 'R1': '$R_1$'}[stat])
    axes[r, 0].set_ylabel('proportion of dances')
plt.tight_layout()
plt.show()

## Fit the models

In [ ]:
idatas = {}
for key, model in models.items():
    print(f"\n===== {model_labels[key]} =====")
    idatas[key] = fit_state_model(model, key)

## Check convergence heuristics
Because reversals can only be described by states, the chains should agree ($\hat{R} < 1.01$) without relabelling.

**Chain screening** (with `data_inits = True`, flagged chains are only reported, not dropped; the aim is for all chains to agree): in test fits, occasional chains settled during warm-up in poor local optima (e.g. a 'forward' direction near the bound for one condition), with a mean log posterior density about 100 below the other chains.
Such chains carry negligible posterior mass but would distort the summaries, so `fit_state_model` runs 8 chains and drops any whose mean log posterior is more than `logp_threshold` (20) below the best chain, reporting which were dropped.
The remaining chains should then agree; if they do not, or if many chains are dropped, the posterior is genuinely multimodal and the model needs revising rather than more screening.
(Chain stacking, Yao, Vehtari & Gelman 2022, *Bayesian Analysis* 17(4), would be a more formal alternative, but requires reliable dance-level LOO, which is not available here; see *Model comparison*.)

In [ ]:
idata = idatas['three_states_condition']
main_vars = ['mu_condition', 'mu_effect', 'kappa_Intercept', 'kappa_effect',
             'reversed_Intercept', 'reversed_effect', 'axial_Intercept', 'axial_effect',
             'w_axial', 'unrelated_Intercept', 'unrelated_effect', 'disoriented_Intercept', 'mu_Individual_sigma', 'mu_Dance_sigma',
             'kappa_Individual_sigma', 'kappa_Dance_sigma',
             'reversed_Individual_sigma', 'axial_Individual_sigma']
main_vars = [v for v in main_vars if v in idata.posterior]
print('Divergences per chain:', idata.sample_stats['diverging'].sum('draw').values)
az.summary(idata, var_names=main_vars, circ_var_names=['mu_condition'])

In [ ]:
az.plot_trace(idata, var_names=main_vars)
plt.tight_layout()
plt.show()

In [ ]:
# Convergence across all models, for the parameters each one contains
for key, idata_k in idatas.items():
    vs = [v for v in main_vars if v in idata_k.posterior]
    summ = az.summary(idata_k, var_names=vs)
    print(f"{model_labels[key]}: divergences {int(idata_k.sample_stats['diverging'].sum())}, "
          f"max R-hat {summ['r_hat'].max():.3f}, min ESS bulk {summ['ess_bulk'].min():.0f}")

## Compare kept and dropped chains
Chain means of the main parameters for every chain (before screening), with each chain's mean log posterior density.
Parameters that differ between dropped and kept chains show which configurations the dropped chains settled in.
Compare these with the prior predictive check above: if those configurations are ones the priors allow but you would have ruled out beforehand, that is a reason to tighten the priors.

In [ ]:
def chain_table(idata_all, threshold=logp_threshold):
    """Chain means of the main parameters, for all chains (columns), marking which were kept."""
    lp_name = 'logp' if 'logp' in idata_all.sample_stats else 'lp'
    lp = idata_all.sample_stats[lp_name].mean('draw').values
    post = idata_all.posterior
    tab = {'mean log posterior': lp.round(1), 'kept': lp >= lp.max() - threshold}
    for c, cond in enumerate(conditions_incl):
        z = np.exp(1j * post['mu_condition'].isel(condition=c)).mean('draw')
        tab[f'mu {cond} (°)'] = np.rad2deg(np.angle(z.values)).round(0)
    for v in ['mu_Individual_sigma', 'mu_Dance_sigma', 'kappa_Intercept', 'w_axial',
              'reversed_Intercept', 'reversed_effect', 'axial_Intercept', 'axial_effect', 'unrelated_Intercept', 'unrelated_effect', 'disoriented_Intercept']:
        if v not in post:
            continue
        x = post[v]
        if 'effect' in x.dims:
            for e in x['effect'].values:
                tab[f'{v}[{e}]'] = x.sel(effect=e).mean('draw').values.round(2)
        else:
            tab[v] = x.mean('draw').values.round(2)
    off = np.abs(wrap(post['mu_offset'].values)) # chain x draw x dance
    tab['dances |offset| > 45°'] = (off > np.pi / 4).mean(axis=(1, 2)).round(3)
    return pd.DataFrame(tab).T.rename(columns=lambda c: f'chain {c}')

for key, idata_all in idatas_all.items():
    print(f"\n===== {model_labels[key]} =====")
    display(chain_table(idata_all))

# Model comparison
Leave-one-**dance**-out cross-validation (PSIS-LOO on the dance-level log-likelihood), which asks how well each model predicts a new dance.

*N.B.* Each dance has its own direction, concentration and state, so leaving one out changes the posterior substantially and PSIS is often unreliable (in test fits, most dances had Pareto $\hat{k} > 0.7$). Treat this comparison as provisional: K-fold cross-validation, refitting with groups of dances left out, would be needed for a reliable comparison.

At low elevation, the `Bambi` models from `HS_circGLMM_mixture_lowel.ipynb` are included if they have been fitted, after summing their run-level log-likelihoods within each dance (their data must match row-for-row).

In [ ]:
import matplotlib as mpl

# Ensure text is output as editable TrueType text elements in PDFs
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'Computer Modern Sans Serif']

loo_states = {model_labels[k]: az.loo(v) for k, v in idatas.items()}

# Bambi models, summed within dances
bambi_files = {
    'Bambi unimodal': '2Results/unwrap_models/HS_lowel_VH_0_unimodal.nc',
    'Bambi axial mixture, same mixing': '2Results/unwrap_models/HS_lowel_VH_1_axial_w1.nc',
    'Bambi axial mixture, condition-dependent mixing': '2Results/unwrap_models/HS_lowel_VH_2_axial_wH.nc',
}
if elevation == 'low' and os.path.exists('0Data/hs_dance_lowel_VH.feather'):
    df_bambi = pd.read_feather('0Data/hs_dance_lowel_VH.feather')
    # dance of each Bambi observation, as the codes used here
    dance_code = dict(zip(dances['Dance_label'], dances['Dance']))
    bambi_dance = df_bambi['Dance_label'].map(dance_code).values
    for name, path in bambi_files.items():
        if not os.path.exists(path) or np.isnan(bambi_dance.astype(float)).any():
            continue
        idata_b = az.from_netcdf(path)
        ll = idata_b.log_likelihood['angle'] # chain x draw x obs
        if ll.sizes['__obs__'] != len(bambi_dance):
            continue
        ll_dance = ll.groupby(xr.DataArray(bambi_dance, dims='__obs__', name='dance')).sum()
        # arviz needs a posterior group (for the relative efficiency), so keep one of the model's parameters
        loo_states[name] = az.loo(az.InferenceData(posterior=idata_b.posterior[['Intercept']],
                                                   log_likelihood=xr.Dataset({'loglik_dance': ll_dance})))

if len(loo_states) > 1:
    comparison = az.compare(loo_states, ic='loo')
    print(comparison.round(2))
for name, loo_k in loo_states.items():
    print(f"{name}: {(loo_k.pareto_k > 0.7).sum().item()} of {loo_k.n_data_points} dances with Pareto k > 0.7")

if len(loo_states) > 1:
    ax_compare = az.plot_compare(comparison)
    ax_compare.get_figure().savefig(f"Plots/HS_states_{elevation}_{fit_tag}_model_comparison.pdf", format="pdf", bbox_inches="tight")

# State probabilities by condition
Population-level probability of each state in each condition, for a typical individual (individual effects at zero), from the condition-dependent three-state model.
Contrasts are relative to the vertical condition and between successive conditions.

In [ ]:
from scipy.special import softmax

post = idatas['three_states_condition'].posterior
n_samp = post.sizes['chain'] * post.sizes['draw']

def draws(name):
    """Posterior draws flattened to (samples, ...)."""
    da = post[name]
    return da.values.reshape(n_samp, *da.shape[2:])

# state logits for each condition: samples x conditions x states (outlier state last, with its own probability)
main_states = [s for s in STATES if s not in OUTLIER_STATES]
eta = np.zeros((n_samp, len(conditions_incl), len(main_states)))
for j, s in enumerate(main_states[1:], start=1):
    eta[:, :, j] = draws(f'{s}_Intercept')[:, None]
    if effect_names:
        eta[:, :, j] += draws(f'{s}_effect') @ X_cond.T
p_cond = softmax(eta, axis=2)
if len(STATES) > len(main_states):
    eta_out = draws(f'{STATES[-1]}_Intercept')[:, None] * np.ones((1, len(conditions_incl)))
    if effect_names and f'{STATES[-1]}_effect' in post:
        eta_out = eta_out + draws(f'{STATES[-1]}_effect') @ X_cond.T
    p_out = (1 / (1 + np.exp(-eta_out)))[:, :, None]
    p_cond = np.concatenate([p_cond * (1 - p_out), p_out], axis=2)

def q(x, digits=3):
    return np.round(np.percentile(x, [50, 2.5, 97.5]), digits)

rows = []
for c, cond in enumerate(conditions_incl):
    for j, s in enumerate(STATES):
        rows.append({'condition': cond, 'state': s, **dict(zip(['median', 'lower', 'upper'], q(p_cond[:, c, j])))})
state_table = pd.DataFrame(rows)
print(state_table.pivot(index='condition', columns='state', values='median').loc[conditions_incl])
state_table

In [ ]:
# Contrasts in state probabilities
contrast_pairs = [(c, 'vertical') for c in conditions_incl[1:]]
contrast_pairs += [(conditions_incl[k], conditions_incl[k - 1]) for k in range(2, len(conditions_incl))]
rows = []
for c1, c0 in contrast_pairs:
    i1, i0 = conditions_incl.index(c1), conditions_incl.index(c0)
    for j, s in enumerate(STATES):
        d = p_cond[:, i1, j] - p_cond[:, i0, j]
        rows.append({'contrast': f'{c1} - {c0}', 'state': s,
                     **dict(zip(['median', 'lower', 'upper'], q(d))),
                     'P(>0)': np.round(np.mean(d > 0), 3)})
contrast_table = pd.DataFrame(rows)
contrast_table

In [ ]:
fig, ax = plt.subplots(figsize=(2 + 2 * len(conditions_incl), 4.5))
width = 0.75 / len(STATES)
for j, s in enumerate(STATES):
    med = np.median(p_cond[:, :, j], axis=0)
    lo, hi = np.percentile(p_cond[:, :, j], [2.5, 97.5], axis=0)
    xs = np.arange(len(conditions_incl)) + (j - (len(STATES) - 1) / 2) * width
    ax.bar(xs, med, width=width, color=state_cols[s], alpha=0.8, label=s)
    ax.errorbar(xs, med, yerr=[med - lo, hi - med], fmt='none', ecolor='black', capsize=3, lw=1)
    # observed proportions of dances assigned to each state (posterior mean probability)
    p_obs = post['p_state'].sel(state=s).mean(('chain', 'draw')).values
    obs_prop = [p_obs[dances['condition'].values == cond].mean() for cond in conditions_incl]
    ax.scatter(xs, obs_prop, color='white', edgecolor='black', zorder=3, s=25)
ax.set_xticks(np.arange(len(conditions_incl)))
ax.set_xticklabels(conditions_incl)
ax.set_ylabel('Probability of state')
ax.set_ylim(0, 1)
ax.set_title(f'Dance states, {elevation} solar elevation\n(bars: typical individual, 95% CI; points: mean over observed dances)')
ax.legend()
fig.savefig(f"Plots/HS_states_{elevation}_{fit_tag}_state_probabilities.pdf", format="pdf", bbox_inches="tight")
plt.show()

## Dance-level state assignments
Posterior probability of each state for every dance, against its unimodal ($R_1$) and axial ($R_2$) mean resultant lengths.

In [ ]:
p_state = post['p_state'].mean(('chain', 'draw')).to_pandas()
dances['state_map'] = p_state.idxmax(axis=1).values
dances['state_prob'] = p_state.max(axis=1).values
for s in STATES:
    dances[f'p_{s}'] = p_state[s].values

print(pd.crosstab(dances['condition'], dances['state_map']))
print('Dances assigned with probability < 0.8:', int((dances['state_prob'] < 0.8).sum()), 'of', n_dance)

fig, axes = plt.subplots(1, len(conditions_incl), figsize=(4 * len(conditions_incl), 4), sharex=True, sharey=True)
for ax, cond in zip(np.atleast_1d(axes), conditions_incl):
    dc = dances[dances['condition'] == cond]
    # colour by state probabilities (RGB mix of the three state colours)
    rgb = sum(np.outer(dc[f'p_{s}'], mpl.colors.to_rgb(state_cols[s])) for s in STATES)
    ax.scatter(dc['R1'], dc['R2'], c=np.clip(rgb, 0, 1), s=15 + 4 * dc['n_runs'], alpha=0.8, edgecolor='gray')
    ax.plot([0, 1], [0, 1], color='gray', lw=0.5)
    ax.set_title(cond)
    ax.set_xlabel('$R_1$ (unimodal)')
np.atleast_1d(axes)[0].set_ylabel('$R_2$ (axial)')
handles = [mpl.lines.Line2D([0], [0], marker='o', color='w', markerfacecolor=state_cols[s], markersize=8, label=s)
           for s in STATES]
np.atleast_1d(axes)[-1].legend(handles=handles, loc='lower right', fontsize='small')
plt.suptitle('Posterior state of each dance (point size: number of runs)')
plt.tight_layout()
plt.show()

# Direction and concentration
Population-level direction of the forward state, $\mu$ (the reversed state points at $\mu + 180°$), and concentration, $\kappa$, for each condition.

In [ ]:
mu_c = draws('mu_condition')
kappa_c = np.log1p(np.exp(draws('kappa_Intercept')[:, None] +
                          (draws('kappa_effect') @ X_cond.T if effect_names else 0)))

def circ_q(x):
    """Circular median and 95% interval in degrees, centred on the circular mean."""
    m = np.angle(np.exp(1j * x).mean())
    return np.round(np.rad2deg(np.percentile(np.angle(np.exp(1j * (x - m))), [50, 2.5, 97.5]) + m))

dir_table = pd.DataFrame(
    [{'condition': cond,
      **dict(zip(['mu_median', 'mu_lower', 'mu_upper'], circ_q(mu_c[:, c]))),
      **dict(zip(['kappa_median', 'kappa_lower', 'kappa_upper'], q(kappa_c[:, c], 2)))}
     for c, cond in enumerate(conditions_incl)])
dir_table

In [ ]:
theta_grid = np.linspace(-np.pi, np.pi, 361)
from scipy.stats import vonmises

fig, axes = plt.subplots(1, len(conditions_incl), figsize=(4 * len(conditions_incl), 4),
                         subplot_kw={'projection': 'polar'})
for ax, (c, cond) in zip(np.atleast_1d(axes), enumerate(conditions_incl)):
    ax.set_theta_direction(-1)
    ax.set_theta_zero_location('N')
    ax.hist(df.loc[df['condition'] == cond, 'angle'], bins=np.linspace(-np.pi, np.pi, 37),
            density=True, color=cond_cols[cond], alpha=0.3)
    # population predictive density: mixture of the states for a typical individual
    # (the outlier state, unrelated or disoriented, is uniform at the population level)
    idx = np.random.default_rng(1).choice(n_samp, 400, replace=False)
    dens = []
    for k in idx:
        f = vonmises.pdf(theta_grid, kappa_c[k, c], loc=mu_c[k, c])
        r = vonmises.pdf(theta_grid, kappa_c[k, c], loc=mu_c[k, c] + np.pi)
        pk = p_cond[k, c]
        wk = draws('w_axial')[k]
        dens.append(pk[0] * f + pk[1] * r + pk[2] * (wk * f + (1 - wk) * r)
                    + (pk[3] / (2 * np.pi) if len(pk) > 3 else 0))
    dens = np.array(dens)
    ax.plot(theta_grid, np.median(dens, axis=0), color=cond_cols[cond], lw=2)
    ax.fill_between(theta_grid, *np.percentile(dens, [2.5, 97.5], axis=0), color=cond_cols[cond], alpha=0.2)
    ax.set_title(cond, pad=15)
    ax.set_yticklabels([])
plt.suptitle(f'Waggle-run angles and population predictive density, {elevation} solar elevation')
plt.tight_layout()
fig.savefig(f"Plots/HS_states_{elevation}_{fit_tag}_densities.pdf", format="pdf", bbox_inches="tight")
plt.show()

# Notes
- **Short dances**: dances with few runs carry little information about their state; their state probabilities stay close to the population-level probabilities for their condition, and this uncertainty is propagated into the condition effects.
- **Large offsets**: a dance whose direction is close to 60° from its condition's direction lies near the bound; check `mu_offset` for values near $\pm 60°$. If many dances sit there, the bound may be too tight for these data.
- **Rotated dances**: at high elevation, many dances are unimodal but rotated 45–135° from the feeder. These are described by direction offsets (up to 60°) within the forward or reversed state, so large rotations widen the spread of individual and dance offsets, and rotations beyond 60° cannot be fully described.
- **Outlier state**: `outlier_state` adds a state for the few dances whose directions are unrelated to the feeder (e.g. solar ephemeris failures). `'unrelated'` suits tightly clustered dances in the wrong direction; `'disoriented'` (uniform) suits diffuse dances, and trades off with low concentration. For a short dance, the unrelated state fits almost as well as forward, so its posterior state probabilities stay close to the prior.